# 2 Engenharia de variáveis (feature engineering)
## Fonte
- `data/[aeroporto]_metar_[ano].parquet`: METAR decodificado no notebook `0_1`.
- `data/[aeroporto]_cgna_[ano].parquet`: movimentos de Congonhas, voo a voo, baixado pelo notebook `0_2`.


In [46]:
# pip install pandas
# pip install numpy
# pip install pyarrow

In [47]:
import numpy as np
import pandas as pd

### Parâmetros

In [48]:
ANO = 2025
AEROPORTO = "cgh"
TZ = "America/Sao_Paulo"

SRC_CGNA = f"data/{str.lower(AEROPORTO)}_cgna_{ANO}.parquet"
SRC_METAR = F"data/{str.lower(AEROPORTO)}_metar_{ANO}.parquet"

OUT = f"data/{str.lower(AEROPORTO)}_taxi_features_{ANO}.parquet"

# Largura do bloco de horário, em minutos.
BLOCO_MIN = 30

# Rumo magnético das cabeceiras e declinação magnética (graus para oeste).
# O vento do METAR vem em graus verdadeiros.
QFU_MAG = {"17": 169, "35": 349}
DECL_W = 21

# Posições e tipos de aeronave com menos partidas que isto viram uma categoria só.
MIN_PARTIDAS = 50

# Companhia e data da mudança no registro da desocupação da posição (notebook 2).
REGIME_CIA = "GLO"
REGIME_MUDANCA = "2025-02-01"

# Registro Aeronáutico Brasileiro (RAB).
RAB_URL = "https://sistemas.anac.gov.br/dadosabertos/Aeronaves/RAB/dados_aeronaves.csv"

## 2.0 Leitura e janela de plausibilidade

A janela tira registros inconsistentes, não valores atípicos. A decolagem antes da
desocupação da posição e as repetições já saíram no download.

In [49]:
cgna = pd.read_parquet(SRC_CGNA)
metar = pd.read_parquet(SRC_METAR)

print(f"partidas no arquivo: {len(cgna)}")
print(f"mensagens METAR e SPECI: {len(metar)}")

partidas no arquivo: 92508
mensagens METAR e SPECI: 11410


In [50]:
# Janela de plausibilidade do tempo de táxi de saída, em minutos.
TAXI_MIN = 3
TAXI_MAX = 90

In [51]:
# Fica só quem está entre TAXI_MIN e TAXI_MAX minutos.
dentro_da_janela = (cgna["taxi_out"] >= TAXI_MIN) & (cgna["taxi_out"] <= TAXI_MAX)
print(f"fora da janela: {len(cgna) - dentro_da_janela.sum()}")
fora_da_janela = cgna[~dentro_da_janela].copy()
cgna = cgna[dentro_da_janela].copy()
cgna["log_taxi"] = np.log(cgna["taxi_out"])
print(f"partidas para análise: {len(cgna)}")
print("--------------------------------")
print(f"partidas fora da janela: {fora_da_janela.describe()}")

fora da janela: 27
partidas para análise: 92481
--------------------------------
partidas fora da janela:         taxi_out
count  27.000000
mean    1.409877
std     0.817468
min     0.166667
25%     0.808333
50%     1.316667
75%     2.066667
max     2.866667


## 2.1 Junção METAR → partida

Cada partida recebe a mensagem, METAR ou SPECI, mais próxima da hora de
desocupação da posição, antes ou depois dela, com no máximo 120 minutos de
diferença. As variáveis meteorológicas medem o tempo que fazia quando o táxi
começou. A tripulação conhece essa condição pela observação direta e pelo ATIS, e a
mensagem mais próxima é a que melhor a representa. É também o que o piloto informa
na ferramenta: o tempo daquele momento.

In [52]:
# Códigos de chuva no tempo presente do METAR. Moderada e forte ficam juntas.
CHUVA_LEVE = ["-DZ", "DZ", "+DZ", "-RA", "-TSRA"]
CHUVA_MOD_FORTE = ["RA", "SHRA", "TSRA", "+RA", "+TSRA"]

# Horas de memória da chuva moderada ou forte.
CHUVA_MEMORIA_H = 4

# Diferença máxima, antes ou depois, entre a mensagem METAR e a desocupação da posição, em minutos.
JANELA_METAR_MIN = 120

In [53]:
# O horário do METAR vem em UTC, sem fuso declarado.
metar = metar.sort_index()
metar["dt"] = metar.index.tz_localize("UTC")

In [54]:
# Marca com 1 as mensagens com chuva moderada ou forte, olhando os três grupos de tempo presente.
metar["chuva_mod_forte"] = 0
for coluna in ["current_wx1", "current_wx2", "current_wx3"]:
    tem_chuva = metar[coluna].isin(CHUVA_MOD_FORTE)
    metar.loc[tem_chuva, "chuva_mod_forte"] = 1

print(f"mensagens com chuva moderada ou forte: {metar['chuva_mod_forte'].sum()}")

# Percorre as mensagens em ordem. Para cada uma, olha quando foi a última chuva
# moderada ou forte ANTES dela. A própria mensagem só é guardada depois.
memoria = pd.Timedelta(hours=CHUVA_MEMORIA_H)
ultima_chuva = None
chuva_anterior = []

for momento, choveu in zip(metar["dt"], metar["chuva_mod_forte"]):
    if ultima_chuva is not None and momento - ultima_chuva < memoria:
        chuva_anterior.append(1)
    else:
        chuva_anterior.append(0)
    if choveu == 1:
        ultima_chuva = momento

metar["chuva_anterior"] = chuva_anterior
print(f"mensagens com chuva nas {CHUVA_MEMORIA_H} h anteriores: {metar['chuva_anterior'].sum()}")

mensagens com chuva moderada ou forte: 239
mensagens com chuva nas 4 h anteriores: 750


In [55]:
COLUNAS_METAR = ["dt", "current_wx1", "current_wx2", "current_wx3",
                 "low_cloud_type", "low_cloud_level",
                 "medium_cloud_type", "medium_cloud_level",
                 "high_cloud_type", "high_cloud_level",
                 "highest_cloud_type", "highest_cloud_level",
                 "wind_direction", "wind_speed", "wind_gust",
                 "cb", "tcu", "chuva_anterior"]

# O merge_asof pede os dois horários na mesma precisão.
cgna["aobt_us"] = cgna["AOBT"].astype("datetime64[us, UTC]")
metar["dt"] = metar["dt"].astype("datetime64[us, UTC]")

cgna = cgna.sort_values("aobt_us")
metar = metar.sort_values("dt")

# Para cada partida, a mensagem mais próxima, antes ou depois, em até JANELA_METAR_MIN minutos.
base = pd.merge_asof(cgna, metar[COLUNAS_METAR],
                     left_on="aobt_us", right_on="dt",
                     direction="nearest",
                     tolerance=pd.Timedelta(minutes=JANELA_METAR_MIN))

In [56]:
sem_metar = base["dt"].isna()
print(f"partidas sem mensagem a menos de {JANELA_METAR_MIN} min: {sem_metar.sum()}")

base = base[~sem_metar].copy()
print(f"partidas com mensagem: {len(base)}")

partidas sem mensagem a menos de 120 min: 0
partidas com mensagem: 92481


## 2.2 Calendário

As variáveis de calendário usam a hora de desocupação da posição no horário de
Brasília. Mês e dia da semana entram como categorias. O horário é agrupado em
blocos de 30 minutos. O primeiro bloco é o das 05h30, com as partidas que deixam a
posição pouco antes da abertura do aeroporto. O último é o das 22h30.

Só entram os feriados em dia útil. São duas variáveis: o próprio feriado e o dia seguinte, que
muitas vezes é sexta-feira de emenda. A véspera e o domingo seguinte foram
testados e não se diferenciaram de um dia comum.

Fontes dos feriados de 2025:
- Nacionais e pontos facultativos: Portaria MGI nº 9.783, de 27/12/2024
  <https://www.gov.br/previc/pt-br/acesso-a-informacao-1/institucional/normas/portarias-1/2025/portaria-mgi-no-9-783-de-27-de-dezembro-de-2024.pdf>
- Estado de São Paulo (9 de julho): Lei estadual nº 9.497, de 05/03/1997
  <https://www.al.sp.gov.br/repositorio/legislacao/lei/1997/lei-9497-05.03.1997.html>
- Cidade de São Paulo (25 de janeiro e Corpus Christi): Lei municipal nº 14.485, de 19/07/2007
  <https://legislacao.prefeitura.sp.gov.br/lei-20000-de-19-de-julho-de-2007>

## 2.2.1 Dia da Semana e Mês

In [57]:
local = base["AOBT"].dt.tz_convert(TZ)

base["data"] = local.dt.date.astype(str)
base["mes"] = local.dt.month

### 2.2.1.1 - Dia da semana

In [58]:
DIAS = {"Monday": "Segunda", "Tuesday": "Terça", "Wednesday": "Quarta",
        "Thursday": "Quinta", "Friday": "Sexta", "Saturday": "Sábado", "Sunday": "Domingo"}

dia_em_ingles = local.dt.day_name()
dia_em_portugues = dia_em_ingles.map(DIAS)
base["dia_semana"] = pd.Categorical(dia_em_portugues, categories=list(DIAS.values()))

print(base["dia_semana"].value_counts(sort=False).to_dict())

{'Segunda': 14111, 'Terça': 14163, 'Quarta': 14312, 'Quinta': 14159, 'Sexta': 14084, 'Sábado': 9446, 'Domingo': 12206}


### 2.2.2 - Bloco de horarios

In [59]:
# Bloco de horário: a hora arredondada para baixo em blocos de BLOCO_MIN minutos.
bloco = local.dt.floor(f"{BLOCO_MIN}min").dt.strftime("%H:%M")

# Depois das 23h00 são poucas partidas, que saíram com atraso.
# Elas entram no último bloco, o das 22h30.
depois_das_23h = local.dt.hour >= 23
bloco[depois_das_23h] = "22:30"
base["bloco"] = bloco

print(f"blocos: {base['bloco'].nunique()}")
print(f"primeiro bloco: {bloco.min()} | último bloco: {bloco.max()}")
print(f"partidas depois das 23h00, levadas para o bloco das 22h30: {depois_das_23h.sum()}")

blocos: 35
primeiro bloco: 05:30 | último bloco: 22:30
partidas depois das 23h00, levadas para o bloco das 22h30: 32


### 2.2.3 - Feriados

In [60]:
FERIADOS = {
    "2025-01-01": "Confraternização Universal",
    "2025-01-25": "Aniversário de São Paulo",       # municipal
    "2025-03-03": "Carnaval",                       # ponto facultativo federal
    "2025-03-04": "Carnaval",                       # ponto facultativo federal
    "2025-04-18": "Paixão de Cristo",
    "2025-04-21": "Tiradentes",
    "2025-05-01": "Dia do Trabalho",
    "2025-06-19": "Corpus Christi",                 # feriado municipal em São Paulo
    "2025-07-09": "Revolução Constitucionalista",   # estadual
    "2025-09-07": "Independência",
    "2025-10-12": "Nossa Senhora Aparecida",
    "2025-11-02": "Finados",
    "2025-11-15": "Proclamação da República",
    "2025-11-20": "Consciência Negra",
    "2025-12-25": "Natal",
}

feriados = pd.to_datetime(list(FERIADOS.keys()))

# Só os feriados de segunda a sexta (dayofweek de 0 a 4).
feriados_uteis = feriados[feriados.dayofweek < 5]
print(f"feriados no ano: {len(feriados)}")
print(f"feriados em dia útil: {len(feriados_uteis)}")

data = pd.to_datetime(base["data"])
ontem = data - pd.Timedelta(days=1)

eh_feriado = data.isin(feriados_uteis)
base["feriado"] = eh_feriado.astype(int)

# Dia seguinte a feriado, desde que o próprio dia não seja feriado
# (a terça de Carnaval é feriado, e não dia seguinte da segunda).
eh_dia_seguinte = ontem.isin(feriados_uteis) & ~eh_feriado
base["dia_seguinte_feriado"] = eh_dia_seguinte.astype(int)

print(f"partidas em feriado: {base['feriado'].sum()}")
print(f"partidas no dia seguinte a feriado: {base['dia_seguinte_feriado'].sum()}")

feriados no ano: 15
feriados em dia útil: 10
partidas em feriado: 2348
partidas no dia seguinte a feriado: 2096


## 2.3 Meteorologia

A chuva tem três faixas: sem, leve e moderada ou forte. A forte foi juntada à
moderada porque a operação com chuva forte é proibida em Congonhas (IS 121-020 da
ANAC, item D4.7.4).

O teto é a base da camada mais baixa com BKN ou OVC. Os cortes são 1.500 pés, o
mínimo do voo visual e 500 pés, abaixo do qual as chegadas
deixam a aproximação RNAV (mínimo de 450 pés) e dependem do ILS (mínimo de 200 pés).

Trovoada, TCU e rajada são fenômenos de tempo significativo e entram como variáveis
binárias. O CB não entra separado, porque a trovoada é sempre produzida por um
cumulonimbus. 

### 2.3.1 - Chuva Leve, Moderada e Forte

In [61]:
# Chuva de cada partida, olhando os três grupos de tempo presente.
chuva_leve = pd.Series(False, index=base.index)
chuva_mod_forte = pd.Series(False, index=base.index)

for coluna in ["current_wx1", "current_wx2", "current_wx3"]:
    chuva_leve = chuva_leve | base[coluna].isin(CHUVA_LEVE)
    chuva_mod_forte = chuva_mod_forte | base[coluna].isin(CHUVA_MOD_FORTE)

chuva = pd.Series("sem", index=base.index)
chuva[chuva_leve] = "leve"
chuva[chuva_mod_forte] = "moderada_forte"
base["chuva"] = pd.Categorical(chuva, categories=["sem", "leve", "moderada_forte"])

print(base["chuva"].value_counts().to_dict())

{'sem': 87834, 'leve': 3978, 'moderada_forte': 669}


### 2.3.2 - Teto

In [62]:
# Teto: a menor altura entre as camadas com BKN ou OVC. FEW e SCT não formam teto.
CAMADAS = [("low_cloud_type", "low_cloud_level"),
           ("medium_cloud_type", "medium_cloud_level"),
           ("high_cloud_type", "high_cloud_level"),
           ("highest_cloud_type", "highest_cloud_level")]

teto_ft = pd.Series(np.nan, index=base.index)
for tipo_nuvem, altura_nuvem in CAMADAS:
    forma_teto = base[tipo_nuvem].isin(["BKN", "OVC"])
    altura = base[altura_nuvem].where(forma_teto)
    # np.fmin fica com o menor valor e ignora o vazio.
    teto_ft = np.fmin(teto_ft, altura)

# Sem camada BKN ou OVC conta como céu alto, na faixa de referência.
teto = pd.Series(">=1500", index=base.index)
teto[teto_ft < 1500] = "500_1500"
teto[teto_ft < 500] = "<500"
base["teto"] = pd.Categorical(teto, categories=[">=1500", "500_1500", "<500"])

print(base["teto"].value_counts().to_dict())

{'>=1500': 72497, '500_1500': 19287, '<500': 697}


### 2.3.3 - Tempo Significativo (WX SIGNIFICANT)

In [63]:
# Trovoada: código TS em qualquer grupo de tempo presente.
trovoada = pd.Series(False, index=base.index)
for coluna in ["current_wx1", "current_wx2", "current_wx3"]:
    tem_ts = base[coluna].astype(str).str.contains("TS")
    trovoada = trovoada | tem_ts
base["trovoada"] = trovoada.astype(int)

base["tcu"] = base["tcu"].astype(int)
base["rajada"] = base["wind_gust"].notna().astype(int)
base["chuva_anterior"] = base["chuva_anterior"].astype(int)

print(f"trovoada: {base['trovoada'].sum()}")
print(f"TCU: {base['tcu'].sum()}")
print(f"rajada: {base['rajada'].sum()}")
print(f"chuva nas {CHUVA_MEMORIA_H} h anteriores: {base['chuva_anterior'].sum()}")

trovoada: 1040
TCU: 1695
rajada: 1588
chuva nas 4 h anteriores: 3646


## 2.4 Pista

A cabeceira em uso é a pista registrada pelo CGNA, sem o L ou R, porque o que
importa ao táxi é o sentido de operação.

Uma troca de cabeceira é contada quando três partidas seguidas saem pela cabeceira
oposta. Assim um registro isolado não cria uma troca que não houve. A variável
marca as partidas que deixaram a posição até 60 minutos depois da troca, enquanto a
fila se reorganiza.

O vento é classificado em relação à cabeceira em uso: abaixo de 15 nós sem cauda
(referência), de cauda, de 15 a 21 nós, e 21 nós ou mais. Vento de cauda tem limite
próprio em Congonhas, de 6 nós com pista molhada (IS 121-020, item D4.7.3).

In [64]:
# Troca de cabeceira: partidas seguidas que confirmam a troca e minutos marcados depois dela.
TROCA_PARTIDAS = 3
JANELA_TROCA_MIN = 60

### 2.4.1 - Cabeceira 17 ou 35

In [65]:
# "17R" vira "17" e "35L" vira "35".
base["cabeceira"] = base["pista"].astype(str).str[:2]
print(base["cabeceira"].value_counts().to_dict())

{'17': 62909, '35': 29572}


### 2.4.2 - Troca de cabeceira

In [66]:
# A troca é lida na ordem das decolagens, porque a pista registrada é a da decolagem.
base = base.sort_values("ATOT").reset_index(drop=True)
cabeceiras = list(base["cabeceira"])
datas = list(base["ATOT"].dt.tz_convert(TZ).dt.date)

# A cabeceira "atual" só muda quando as TROCA_PARTIDAS partidas seguintes confirmam,
# e recomeça a cada dia pela maioria das primeiras decolagens.
atual = cabeceiras[0]
houve_troca = []

for i in range(len(cabeceiras)):
    novo_dia = i == 0 or datas[i] != datas[i - 1]

    if novo_dia:
        primeiras = cabeceiras[i:i + TROCA_PARTIDAS]
        if primeiras.count("17") >= primeiras.count("35"):
            atual = "17"
        else:
            atual = "35"
        houve_troca.append(False)
    else:
        proximas = cabeceiras[i:i + TROCA_PARTIDAS]
        confirmada = proximas.count(cabeceiras[i]) == TROCA_PARTIDAS
        if cabeceiras[i] != atual and confirmada:
            atual = cabeceiras[i]
            houve_troca.append(True)
        else:
            houve_troca.append(False)

print(f"trocas de cabeceira no ano: {sum(houve_troca)}")

# Marca as partidas que decolaram até JANELA_TROCA_MIN minutos depois da última troca,
# e guarda o instante dessa troca para corrigir o vento mais adiante.
janela_troca = pd.Timedelta(minutes=JANELA_TROCA_MIN)
ultima_troca = None
troca_cabeceira = []
instante_troca = []                                        

for momento, trocou in zip(base["ATOT"], houve_troca):
    if trocou:
        ultima_troca = momento
    if ultima_troca is not None and momento - ultima_troca < janela_troca:
        troca_cabeceira.append(1)
        instante_troca.append(ultima_troca)                
    else:
        troca_cabeceira.append(0)
        instante_troca.append(pd.NaT)                      

# Grava as colunas ainda na ordem do ATOT, para cada valor ficar na sua partida.
base["troca_cabeceira"] = troca_cabeceira
base["instante_troca"] = instante_troca                    

# Agora sim, volta para a ordem de saída da posição.
base = base.sort_values("AOBT").reset_index(drop=True)
print(f"partidas até {JANELA_TROCA_MIN} min depois de uma troca: {base['troca_cabeceira'].sum()}")

trocas de cabeceira no ano: 245
partidas até 60 min depois de uma troca: 3794


### 2.4.3 - Vento em relação à cabeceira de decolagem

Fontes: 
DECEA: <https://ajuda.decea.mil.br/base-de-conhecimento/como-decodificar-o-metar-e-o-speci/>

Publicação CÓDIGOS METEOROLÓGICO: <https://publicacoes.decea.mil.br/publicacao/ICA-105-16>

O METAR/SPECI são confeccionados pelo Norte Verdadeiro. 
Para a correta distinção de vento de proa ou de cauda é necessário fazer o calculo do rumo magnetico da pista -/+ a declinação magnética(quando Oeste(W) é menos, quando Leste(E) é mais). Desta maneira temos o rumo verdadeiro da cabeceira

In [78]:
# Cortes de intensidade do vento, em nós. [10, 20] gera as faixas <10, 10_20 e >=20.
VENTO_CORTES = [10, 20]

In [80]:
# Rumo verdadeiro de cada cabeceira, para comparar com o vento do METAR.
rumo_verdadeiro = {}
for cabeceira in QFU_MAG:
    rumo_verdadeiro[cabeceira] = (QFU_MAG[cabeceira] - DECL_W) % 360

rumo_pista = base["cabeceira"].map(rumo_verdadeiro)

# Componente do vento no eixo da pista: positiva é de proa, negativa é de cauda.
angulo = np.radians(base["wind_direction"] - rumo_pista)
componente_proa = base["wind_speed"] * np.cos(angulo)

# Vento de cauda dentro da janela de uma troca é METAR atrasado: a pista já mudou,
# mas a mensagem ainda mostra o vento de antes.
cauda_no_metar = componente_proa.fillna(0) < 0
base["atrasada"] = cauda_no_metar & (base["troca_cabeceira"] == 1)
print(f"partidas com METAR atrasado: {base['atrasada'].sum()}")

# Para cada partida com METAR atrasado, procura a próxima mensagem com vento de proa
# (ou variável) para a cabeceira da decolagem, até MAX_ESPERA_H horas depois.
MAX_ESPERA_H = 1

base["direcao"] = base["wind_direction"]
base["intensidade"] = base["wind_speed"]
espera_min = []
nao_achou = 0

for indice in base.index[base["atrasada"]]:
    rumo = rumo_pista[indice]
    mensagem_atual = base.loc[indice, "dt"]
    limite = mensagem_atual + pd.Timedelta(hours=MAX_ESPERA_H)
    seguintes = metar[(metar["dt"] > mensagem_atual) & (metar["dt"] <= limite)]

    achou = False
    for momento, direcao, intensidade in zip(seguintes["dt"], seguintes["wind_direction"],
                                             seguintes["wind_speed"]):
        sem_direcao = np.isnan(direcao)
        if sem_direcao or intensidade * np.cos(np.radians(direcao - rumo)) >= 0:
            base.loc[indice, "direcao"] = direcao
            base.loc[indice, "intensidade"] = intensidade
            espera_min.append((momento - mensagem_atual).total_seconds() / 60)
            achou = True
            break

    # Sem mensagem de proa no prazo: a direção é apagada, e a partida não conta como cauda.
    if not achou:
        base.loc[indice, "direcao"] = np.nan
        nao_achou = nao_achou + 1

print(f"acharam mensagem de proa: {len(espera_min)} "
      f"(mediana {np.median(espera_min):.0f} min depois da mensagem original)")
print(f"sem mensagem de proa em {MAX_ESPERA_H} h: {nao_achou}")

# Com o vento corrigido, a conta de proa e cauda é refeita.
angulo = np.radians(base["direcao"] - rumo_pista)
base["componente_proa"] = base["intensidade"] * np.cos(angulo)

# Vento variável ou calmo não tem direção, e conta como sem cauda.
base["de_cauda"] = base["componente_proa"].fillna(0) < 0

# Faixas de intensidade montadas a partir de VENTO_CORTES.
# Ex.: [10, 20] gera "<10", "10_20" e ">=20".
rotulos = [f"<{VENTO_CORTES[0]}"]
for i in range(1, len(VENTO_CORTES)):
    rotulos.append(f"{VENTO_CORTES[i - 1]}_{VENTO_CORTES[i]}")
rotulos.append(f">={VENTO_CORTES[-1]}")

# Classificação do vento: primeiro pela intensidade, depois a cauda por cima.
vento = pd.Series(rotulos[0], index=base.index)
for i, corte in enumerate(VENTO_CORTES):
    vento[base["intensidade"] >= corte] = rotulos[i + 1]
vento[base["de_cauda"]] = "cauda"

# A primeira faixa é a referência do modelo; a cauda vem logo depois dela.
base["vento"] = pd.Categorical(vento, categories=[rotulos[0], "cauda"] + rotulos[1:])
print(base["vento"].value_counts().to_dict())

partidas com METAR atrasado: 636
acharam mensagem de proa: 560 (mediana 60 min depois da mensagem original)
sem mensagem de proa em 1 h: 76
{'<10': 62874, '10_20': 24843, 'cauda': 4299, '>=20': 340}


## 2.5 Posição de estacionamento

Cada posição é uma categoria. No modelo ela entra em interação com a cabeceira,
porque uma posição perto de uma cabeceira fica longe da outra. Em Congonhas a
decolagem só sai das interseções mais extremas de cada cabeceira (IS 121-020, item
D4.7.5), então a distância percorrida depende da posição e da cabeceira.

Quando o registro traz mais de uma posição (aeronave rebocada antes da partida),
vale a primeira. Posições só com número ficam com dois dígitos. As que tiveram menos
de 50 partidas no ano (hangar, pátios remotos, códigos avulsos) viram "OUTRAS". Sem
registro, a partida é descartada.

In [68]:
posicao = base["box"].astype(str).str.strip().str.upper()

# Aeronave rebocada: o registro traz "A;B" e vale a primeira posição.
posicao = posicao.str.split(";").str[0].str.strip()

# Registro vazio fica sem valor.
posicao = posicao.replace({"": np.nan, "NAN": np.nan, "NONE": np.nan})

# "7" e "07" são a mesma posição.
so_numero = posicao.str.isdigit() == True
posicao[so_numero] = posicao[so_numero].str.zfill(2)

partidas_por_posicao = posicao.value_counts()
posicoes_raras = partidas_por_posicao[partidas_por_posicao < MIN_PARTIDAS].index

posicao[posicao.isin(posicoes_raras)] = "OUTRAS"
base["posicao"] = posicao

print(f"posições com {MIN_PARTIDAS} partidas ou mais: {len(partidas_por_posicao) - len(posicoes_raras)}")
print(f"posições agrupadas em OUTRAS: {len(posicoes_raras)}")
print(f"partidas sem posição registrada: {base['posicao'].isna().sum()}")

posições com 50 partidas ou mais: 32
posições agrupadas em OUTRAS: 12
partidas sem posição registrada: 4


## 2.6 Tipo de aeronave

O tipo vem da matrícula, no Registro Aeronáutico Brasileiro (RAB), e não do código
informado no voo, que diverge do registro em cerca de 9% das partidas. O RAB guarda
também as matrículas canceladas, então as aeronaves que já saíram da frota
continuam lá.

O arquivo do RAB traz nomes e documentos dos proprietários. Por isso ele é lido
direto do site da ANAC, só com as colunas necessárias, e não é gravado.

As duas versões do ATR 72 (AT75 e AT76) ficam juntas. Tipos com menos de 50
partidas viram "OUTROS".

In [69]:
rab = pd.read_csv(RAB_URL, sep=";", skiprows=1, encoding="utf-8-sig", dtype=str,
                  usecols=["MARCAS", "CD_TIPO_ICAO", "DS_MODELO"])

# Se uma matrícula aparecer duas vezes, fica o registro mais recente.
rab = rab.drop_duplicates(subset="MARCAS", keep="last")
print(f"matrículas no RAB: {len(rab)}")

matrículas no RAB: 34894


In [70]:
# A matrícula do CGNA vem sem hífen, como no RAB (ex.: PRXBM).
base["matricula"] = base["matricula"].astype(str).str.strip().str.upper()
base["matricula"] = base["matricula"].str.replace("-", "")

base = base.merge(rab, left_on="matricula", right_on="MARCAS", how="left")
print(f"partidas sem matrícula no RAB: {base['CD_TIPO_ICAO'].isna().sum()}")

partidas sem matrícula no RAB: 122


In [71]:
tipo = base["CD_TIPO_ICAO"].copy()

# ATR 72-500 (AT75) e ATR 72-600 (AT76) ficam no mesmo grupo.
tipo[tipo == "AT76"] = "AT75"

partidas_por_tipo = tipo.value_counts()
tipos_raros = partidas_por_tipo[partidas_por_tipo < MIN_PARTIDAS].index
tipo[tipo.isin(tipos_raros)] = "OUTROS"
base["tipo_aeronave"] = tipo

print(base["tipo_aeronave"].value_counts().to_dict())

{'A320': 19789, 'B738': 19447, 'A319': 12148, 'B38M': 11457, 'A20N': 10944, 'B737': 7481, 'E195': 6921, 'E295': 3556, 'AT75': 610, 'OUTROS': 6}


## 3.7 Regime de registro

A série mensal por companhia (notebook 2) mostra que, a partir de 1º de fevereiro de
2025, o tempo de táxi de saída da GOL caiu de uma vez e ficou igual ao das outras
companhias, saindo das mesmas posições para as mesmas cabeceiras, em todos os
aeroportos ao mesmo tempo. Como o VRA é informado pelas próprias companhias, a
mudança combina com uma alteração no momento em que a GOL passou a registrar a
desocupação da posição.

A variável marca as partidas da GOL antes dessa data. Ela corrige uma diferença de
medição, não é um efeito de companhia, e vale zero em qualquer partida depois dela.

In [72]:
local = base["AOBT"].dt.tz_convert(TZ)
data_mudanca = pd.Timestamp(REGIME_MUDANCA, tz=TZ)

antes_da_mudanca = local < data_mudanca
da_companhia = base["companhia"].astype(str) == REGIME_CIA

base["regime_anterior"] = (antes_da_mudanca & da_companhia).astype(int)
print(f"partidas no regime anterior: {base['regime_anterior'].sum()}")

partidas no regime anterior: 3329


## 3.8 Descartes e gravação

Saem as partidas sem registro de alguma variável (posição vazia ou matrícula fora do
RAB). A companhia fica gravada só para descrição e não entra no modelo.

In [73]:
VARIAVEIS = ["mes", "dia_semana", "bloco", "feriado", "dia_seguinte_feriado",
             "chuva", "teto", "trovoada", "tcu", "rajada", "chuva_anterior",
             "cabeceira", "troca_cabeceira", "vento",
             "posicao", "tipo_aeronave", "regime_anterior"]

antes = len(base)
base = base.dropna(subset=VARIAVEIS).reset_index(drop=True)

print(f"partidas sem registro de alguma variável: {antes - len(base)}")
print(f"base final: {len(base)} partidas em {base['data'].nunique()} dias")
print(f"mediana do tempo de táxi de saída: {base['taxi_out'].median():.1f} min")

partidas sem registro de alguma variável: 125
base final: 92356 partidas em 365 dias
mediana do tempo de táxi de saída: 13.7 min


In [74]:
GRAVAR = ["taxi_out", "log_taxi", "AOBT", "data", "companhia"] + VARIAVEIS
base[GRAVAR].to_parquet(OUT, index=False)
print(f"salvo: {OUT}")

salvo: data/cgh_taxi_features_2025.parquet


In [75]:
# Resumo das variáveis gravadas (a Tabela 1 do TCC).
resumo = []
for coluna in VARIAVEIS:
    niveis = base[coluna].nunique()
    mais_comum = base[coluna].astype(str).value_counts().index[0]
    resumo.append({"variável": coluna, "níveis": niveis, "mais comum": mais_comum})

pd.DataFrame(resumo)

,variável,níveis,mais comum
0,mes,12,10
1,dia_semana,7,Quarta
2,bloco,35,08:30
3,feriado,2,0
4,dia_seguinte_feriado,2,0
5,chuva,3,sem
6,teto,3,>=1500
7,trovoada,2,0
8,tcu,2,0
9,rajada,2,0
